# Notebook 2: Dataset, Augmentation & DataLoader

**What this does (in simple terms):**

Before training a model, we need to build a "data pipeline" — the system
that feeds audio to the model in organized batches.

This notebook:
1. Loads the preprocessed audio files from Notebook 1
2. Builds augmentation (RawBoost + Codec) — adds random noise/distortion
   to make the model tougher and more generalized
3. Creates a PyTorch Dataset — the object that loads one audio at a time
4. Creates DataLoaders — the object that groups audio into batches
5. Tests everything works before we start training

**Run this on Colab right after Notebook 1 (same session).**

## 2.1 Imports & Check Preprocessed Data Exists

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Configuration
PREPROCESSED_DIR = Path("/content/preprocessed")
TARGET_SR = 16000
MAX_DURATION = 6
MAX_SAMPLES = TARGET_SR * MAX_DURATION  # 96000
BATCH_SIZE = 8

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Verify data exists
for split in ["train", "dev", "eval"]:
    folder = PREPROCESSED_DIR / split
    count = len(list(folder.glob("*.flac"))) if folder.exists() else 0
    print(f"{split}: {count} files")

assert (PREPROCESSED_DIR / "metadata.csv").exists(), "metadata.csv not found! Run Notebook 1 first."
metadata = pd.read_csv(PREPROCESSED_DIR / "metadata.csv")
print(f"\nMetadata: {len(metadata)} entries")

## 2.2 Load Protocol Files

Protocol files are like a label sheet — they tell us which audio
is real (bonafide) and which is fake (spoof), and what attack
type was used to create the fake audio.

In [ ]:
def load_protocol(protocol_path):
    """Load ASVspoof protocol file and add integer labels."""
    df = pd.read_csv(
        protocol_path, sep=" ", header=None,
        names=["speaker_id", "audio_id", "system_id", "attack_type", "label"]
    )
    # bonafide = 0 (real), spoof = 1 (fake)
    df["label_int"] = (df["label"] == "spoof").astype(int)
    return df

PROTOCOL_DIR = PREPROCESSED_DIR / "protocols"
train_protocol = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.train.trn.txt")
dev_protocol = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.dev.trl.txt")
eval_protocol = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.eval.trl.txt")

print(f"Train: {len(train_protocol)} samples")
print(f"Dev:   {len(dev_protocol)} samples")
print(f"Eval:  {len(eval_protocol)} samples")
print(f"\nTrain labels:\n{train_protocol['label'].value_counts()}")

## 2.3 RawBoost Augmentation

**What is RawBoost?**
RawBoost simulates real-world audio distortions. Imagine someone
calling you on a bad phone connection — the audio gets distorted
by the channel. RawBoost does this artificially so the model
learns to detect deepfakes even in messy audio conditions.

Two types:
- **Type 1 (Convolutive)**: Simulates channel effects (like different microphones)
- **Type 2 (Impulsive noise)**: Adds noise that depends on the audio signal

In [ ]:
def rawboost_convolutive(audio_np, sr=TARGET_SR):
    """
    Simulates channel effects by applying random notch filters.
    Like recording through different microphones or phone lines.
    """
    n_notch = np.random.randint(1, 5)
    n = len(audio_np)
    spectrum = np.fft.rfft(audio_np)
    freqs = np.fft.rfftfreq(n, d=1.0 / sr)

    for _ in range(n_notch):
        freq = np.random.uniform(100, sr // 2 - 100)
        bw = np.random.uniform(50, 300)
        mask = 1.0 - np.exp(-0.5 * ((freqs - freq) / (bw / 2 + 1e-8)) ** 2)
        spectrum = spectrum * mask

    return np.fft.irfft(spectrum, n=n).astype(np.float32)


def rawboost_impulsive_noise(audio_np, sr=TARGET_SR):
    """
    Adds signal-dependent noise — louder parts get more noise,
    quiet parts get less. More realistic than uniform noise.
    """
    noise_level = np.random.uniform(0.001, 0.01)
    noise = noise_level * np.random.randn(len(audio_np)) * np.abs(audio_np)
    return (audio_np + noise).astype(np.float32)


def rawboost_augment(audio_np):
    """Apply both RawBoost types with 50% chance each."""
    audio_np = audio_np.copy()
    if np.random.rand() < 0.5:
        audio_np = rawboost_convolutive(audio_np)
    if np.random.rand() < 0.5:
        audio_np = rawboost_impulsive_noise(audio_np)
    return audio_np

## 2.4 Codec Augmentation

**What is Codec Augmentation?**
When you send audio over WhatsApp, Zoom, or a phone call,
it gets compressed. This compression changes the audio slightly.
Codec augmentation simulates this so the model can handle
compressed audio from real-world sources.

In [ ]:
def codec_augment(audio_np, sr=TARGET_SR):
    """Simulate audio compression (like WhatsApp/Zoom calls)."""
    audio_t = torch.from_numpy(audio_np).unsqueeze(0).float()

    if np.random.rand() < 0.5:
        # mu-law: simulates telephone compression
        q = torchaudio.functional.mu_law_encoding(audio_t, 256)
        audio_t = torchaudio.functional.mu_law_decoding(q, 256)
    else:
        # Downsample to 8kHz and back: loses high frequencies (telephone bandwidth)
        audio_t = torchaudio.transforms.Resample(sr, 8000)(audio_t)
        audio_t = torchaudio.transforms.Resample(8000, sr)(audio_t)

    return audio_t.squeeze(0).numpy()


def apply_augmentation(audio_tensor):
    """
    Full augmentation pipeline.
    Called on every training sample, every epoch — always random.
    """
    audio_np = audio_tensor.numpy().copy()

    # RawBoost: 70% chance
    if np.random.rand() < 0.7:
        audio_np = rawboost_augment(audio_np)

    # Codec: 30% chance
    if np.random.rand() < 0.3:
        audio_np = codec_augment(audio_np)

    return torch.from_numpy(audio_np).float()

## 2.5 Visualize Augmentation

Let's see what augmentation actually does to an audio file.
The same audio looks different each time because augmentation is random.

In [ ]:
sample_file = list((PREPROCESSED_DIR / "train").glob("*.flac"))[0]
sample_audio, sr = torchaudio.load(str(sample_file))
sample_audio = sample_audio.squeeze(0)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))

axes[0].plot(sample_audio.numpy(), color="steelblue")
axes[0].set_title("Original Audio (preprocessed)")
axes[0].set_ylabel("Amplitude")

for i in range(1, 4):
    augmented = apply_augmentation(sample_audio)
    axes[i].plot(augmented.numpy(), color="coral")
    axes[i].set_title(f"Augmented Version {i} (random each time)")
    axes[i].set_ylabel("Amplitude")

axes[-1].set_xlabel("Samples")
plt.tight_layout()
plt.show()

## 2.6 PyTorch Dataset

**What is a Dataset?**
A Dataset is like a smart filing cabinet. When the model asks for
sample #5000, the Dataset knows how to:
1. Find the right audio file
2. Apply augmentation (if training)
3. Normalize the volume
4. Create the attention mask
5. Return everything in a neat package

In [ ]:
class ASVspoofDataset(Dataset):
    def __init__(self, protocol_df, audio_dir, metadata_df, is_train=False):
        """
        Args:
            protocol_df: DataFrame with audio_id, label, attack_type
            audio_dir: Path to preprocessed audio files
            metadata_df: DataFrame with original_length for each audio
            is_train: If True, apply augmentation
        """
        self.protocol = protocol_df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.is_train = is_train

        # Build lookup: audio_id -> original_length (for creating masks)
        self.length_lookup = {}
        for _, row in metadata_df.iterrows():
            if row["status"] == "success":
                self.length_lookup[row["audio_id"]] = int(row["original_length"])

    def __len__(self):
        return len(self.protocol)

    def __getitem__(self, idx):
        row = self.protocol.iloc[idx]
        audio_id = row["audio_id"]

        # 1. Load preprocessed audio
        file_path = self.audio_dir / f"{audio_id}.flac"
        audio, sr = torchaudio.load(str(file_path))
        audio = audio.squeeze(0)  # [96000]

        # 2. Augmentation (training only, random each time)
        if self.is_train:
            audio = apply_augmentation(audio)

        # 3. Peak normalization (after augmentation)
        max_val = torch.max(torch.abs(audio))
        if max_val > 0:
            audio = audio / max_val

        # 4. Create attention mask from original length
        original_length = self.length_lookup.get(audio_id, MAX_SAMPLES)
        mask = torch.zeros(MAX_SAMPLES)
        mask[:original_length] = 1.0

        return {
            "audio": audio,                                          # [96000]
            "mask": mask,                                             # [96000]
            "label": torch.tensor(row["label_int"], dtype=torch.long),  # 0 or 1
            "audio_id": audio_id,
            "attack_type": row["attack_type"],
        }

## 2.7 Create & Test DataLoaders

In [ ]:
# Create datasets
train_dataset = ASVspoofDataset(
    train_protocol, PREPROCESSED_DIR / "train", metadata, is_train=True
)
dev_dataset = ASVspoofDataset(
    dev_protocol, PREPROCESSED_DIR / "dev", metadata, is_train=False
)
eval_dataset = ASVspoofDataset(
    eval_protocol, PREPROCESSED_DIR / "eval", metadata, is_train=False
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Dev:   {len(dev_dataset)} samples, {len(dev_loader)} batches")
print(f"Eval:  {len(eval_dataset)} samples, {len(eval_loader)} batches")

In [ ]:
# Test one batch
batch = next(iter(train_loader))
print("Batch contents:")
print(f"  audio shape:   {batch['audio'].shape}")       # [8, 96000]
print(f"  mask shape:    {batch['mask'].shape}")         # [8, 96000]
print(f"  labels:        {batch['label'].tolist()}")
print(f"  audio_ids:     {batch['audio_id'][:3]}...")
print(f"  attack_types:  {batch['attack_type'][:3]}...")
print(f"  audio max val: {batch['audio'].abs().max():.4f} (should be ~1.0)")

In [ ]:
# Verify augmentation randomness
a1 = train_dataset[0]["audio"]
a2 = train_dataset[0]["audio"]
diff = torch.abs(a1 - a2).mean().item()
print(f"Same sample loaded twice — difference: {diff:.6f}")
print(f"Augmentation working: {'YES' if diff > 1e-6 else 'NO — check augmentation code!'}")

In [ ]:
print("\n✅ Notebook 2 complete.")
print("Dataset and DataLoader are working correctly.")
print("Proceed to Notebook 3: Teacher Model Training.")